# 04 — Quantum kernel SVM (QSVM)

QSVM maps each reduced feature vector to a quantum state with a ZZ feature map, then uses the
fidelity kernel

$$K(x, x') = |\langle \phi(x) \mid \phi(x') \rangle|^2$$

inside a classical soft-margin SVM. The kernel is computed by **exact statevector simulation**
(`qiskit.quantum_info.Statevector`), so these results are shot-noise free and represent an upper
bound relative to noisy hardware, not a hardware result.

In [ ]:
from pathlib import Path
import json
import sys

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image as PILImage

if hasattr(sys.stdout, "reconfigure"):
    sys.stdout.reconfigure(encoding="utf-8")

ROOT = Path.cwd()
if not (ROOT / "results").exists():
    ROOT = ROOT.parent
RESULTS = ROOT / "results"
DATA = ROOT / "data"
plt.rcParams["figure.figsize"] = (9, 5)
plt.rcParams["axes.grid"] = True


def show_image(path):
    """Render a saved PNG inside the notebook without extra display dependencies."""
    img = PILImage.open(path)
    plt.figure(figsize=(min(12, img.width / 100), min(8, img.height / 100)))
    plt.imshow(img)
    plt.axis("off")
    plt.show()
    plt.close()


print("Project root:", ROOT)

## 1. Circuit and resources

In [ ]:
circuit_text = (RESULTS / "circuits" / "qsvm_circuit.txt").read_text(encoding="utf-8")
print(circuit_text[:2500])

In [ ]:
qres = json.loads((RESULTS / "circuits" / "qsvm_resources.json").read_text(encoding="utf-8"))
qres

Eight qubits, depth 31, 28 two-qubit gates, no trainable circuit parameters. The trainable part is
the classical SVM; the quantum device only evaluates the kernel.

## 2. Build the encoder and inspect a small kernel

In [ ]:
import sys
sys.path.insert(0, str(ROOT))

from src.quantum.encoding import EncodingConfig, build_encoder
from src.quantum.circuits import fidelity_kernel

enc_cfg = EncodingConfig(method="angle", n_features=8, map_name="zz", reps=2, entanglement="linear")
encoder = build_encoder(enc_cfg)
print(enc_cfg.to_dict())

arrays = np.load(DATA / "processed" / "arrays.npz")
X_small = arrays["X_test_common"][:4]
states = encoder.states(X_small)
K_small = fidelity_kernel(states)
np.round(K_small, 4)

Diagonal entries are exactly 1 and the matrix is symmetric positive semidefinite by construction.
The full 600 × 600 training Gram matrix is what the SVM optimises over.

## 3. Qiskit cross-check

In [ ]:
qmeta = json.loads((ROOT / "models" / "qsvm_meta.json").read_text(encoding="utf-8"))
qmeta.get("info", {}).get("kernel_validation", qmeta.get("info", {}))

The exact kernel was cross-checked against Qiskit `FidelityQuantumKernel` with `ComputeUncompute`
on 16 samples and 100,000 shots. The maximum absolute difference was 0.00171 against a tolerance of
0.00632 (4σ shot-noise bound).

## 4. Test performance

In [ ]:
metrics = json.loads((RESULTS / "metrics" / "qsvm.json").read_text(encoding="utf-8"))
keys = ["threshold", "accuracy", "precision", "recall", "f1", "roc_auc", "fpr", "fnr", "mcc",
        "balanced_accuracy", "tn", "fp", "fn", "tp"]
pd.Series({k: metrics[k] for k in keys}).to_frame("QSVM")

In [ ]:
show_image(RESULTS / "figures" / "qsvm_confusion_matrix.png")
show_image(RESULTS / "figures" / "qsvm_roc_curve.png")

QSVM reaches F1 0.9231 and ROC-AUC 0.9903. It has the highest recall (0.9737) but a much higher
false-positive rate (0.1185) than the CNN+LSTM. Its ROC-AUC is 0.0012 above the CNN, but the paired
bootstrap interval crosses zero, so this difference is not resolved.

## 5. Feature sensitivity

In [ ]:
sens = pd.read_csv(RESULTS / "explainability" / "qsvm_feature_sensitivity.csv")
sens.head(10)

In [ ]:
show_image(RESULTS / "figures" / "qsvm_feature_sensitivity.png")

**Takeaways**

1. Exact statevector fidelity kernel, cross-checked against Qiskit's shot-based kernel.
2. Competitive F1 and the best recall, but more false positives than the classical baseline.
3. No quantum advantage is observed; the higher AUC is within noise.
4. Resources: 8 qubits, 28 two-qubit gates, no trainable quantum parameters.